# Data Quality Dashboard

**Automated quality checks for key economic time series from FRED.**

This notebook pulls 13 key economic series, runs validation checks
(NaN patterns, duplicate/non-monotonic dates, outliers, staleness,
frequency gaps), and presents a summary table with a timeline
visualization of data coverage.

*Dependencies:*

- Repository: https://github.com/rsvp/fecon235
- Python: matplotlib, pandas, numpy
- FRED API access via fecon235

In [ ]:
from fecon235.fecon235 import *

from __future__ import absolute_import, print_function
system.specs()

import sys; sys.path.insert(0, '..')
from fecon235.data_validation import (
    validate_series, validate_frequency, check_staleness, validate_all
)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime

pd.set_option('display.notebook_repr_html', True)
%matplotlib inline

## Key Economic Series

We monitor 13 important economic indicators spanning GDP, inflation,
employment, interest rates, equity markets, housing, industrial output,
retail activity, money supply, and trade.

In [ ]:
# Define the series to monitor: (display_name, FRED code, expected_freq)
SERIES_DEFS = [
    ("GDP (Real)",        "GDPC1",    "Q"),
    ("CPI",               "CPIAUCSL", "M"),
    ("Core PCE",          "PCEPILFE", "M"),
    ("Unemployment Rate", "UNRATE",   "M"),
    ("Nonfarm Payrolls",  "PAYEMS",   "M"),
    ("Fed Funds Rate",    "FEDFUNDS", "M"),
    ("10Y Treasury",      "GS10",     "M"),
    ("S&P 500",           "SP500",    "M"),
    ("Housing Starts",    "HOUST",    "M"),
    ("Industrial Prod.",  "INDPRO",   "M"),
    ("Retail Sales",      "RSAFS",    "M"),
    ("M2 Money Supply",   "M2SL",     "M"),
    ("Trade Balance",     "BOPGSTB",  "M"),
]

print(f"Monitoring {len(SERIES_DEFS)} economic series...")

### Fetching Data from FRED

In [ ]:
# Download all series
series_data = {}
fetch_errors = []

for display_name, fred_code, freq in SERIES_DEFS:
    try:
        df = get(fred_code)
        series_data[display_name] = (df, fred_code, freq)
        print(f"  \u2713 {display_name} ({fred_code}): {len(df)} observations")
    except Exception as e:
        fetch_errors.append((display_name, fred_code, str(e)))
        print(f"  \u2717 {display_name} ({fred_code}): FAILED - {e}")

if fetch_errors:
    print(f"\n{len(fetch_errors)} series failed to download.")
else:
    print(f"\nAll {len(SERIES_DEFS)} series downloaded successfully.")

## Validation Results

In [ ]:
# Run all validation checks and build summary
import re

MAX_AGE_DAYS = 90
today = pd.Timestamp.now()

summary_rows = []

for display_name, (df, fred_code, expected_freq) in series_data.items():
    # Extract series
    s = df.iloc[:, 0] if isinstance(df, pd.DataFrame) else df

    # Run combined validation
    result = validate_all(s, name=display_name,
                          expected_freq=expected_freq,
                          max_age_days=MAX_AGE_DAYS)

    # Compute summary statistics
    last_date = s.index[-1] if len(s) > 0 else None
    staleness_days = (today - last_date).days if last_date else None
    nan_count = int(s.isna().sum())
    obs_count = len(s)

    # Count outliers from validate_series warnings
    outlier_count = 0
    for w in result.warnings:
        if "outlier" in w.lower():
            m = re.search(r'(\d+) outlier', w)
            if m:
                outlier_count = int(m.group(1))

    # Infer frequency
    inferred_freq = pd.infer_freq(s.index) if len(s) > 2 else None

    summary_rows.append({
        "Series": display_name,
        "FRED Code": fred_code,
        "Last Update": last_date.strftime("%Y-%m-%d") if last_date else "N/A",
        "Frequency": inferred_freq or "?",
        "Obs Count": obs_count,
        "NaN Count": nan_count,
        "Outliers": outlier_count,
        "Staleness (days)": staleness_days if staleness_days else 0,
        "Status": "PASS" if result.passed and not result.warnings else
                  "FAIL" if not result.passed else "WARN",
        "Details": "; ".join(result.errors + result.warnings) or "OK",
    })

    # Print per-series results
    if result.errors:
        for e in result.errors:
            print(f"\u2717 {display_name}: {e}")
    if result.warnings:
        for w in result.warnings:
            print(f"\u26a0 {display_name}: {w}")
    if result.passed and not result.warnings:
        print(f"\u2713 {display_name}: all checks passed")

summary_df = pd.DataFrame(summary_rows)
print(f"\nValidation complete: {len(summary_df)} series checked.")

### Summary Table

The table below summarizes data quality for all monitored series.
- **PASS** (green): No errors or warnings
- **WARN** (yellow): Warnings but no blocking errors
- **FAIL** (red): Errors detected

In [ ]:
# Display styled summary table
def color_status(val):
    if val == "PASS":
        return "background-color: #d4edda; color: #155724"
    elif val == "WARN":
        return "background-color: #fff3cd; color: #856404"
    elif val == "FAIL":
        return "background-color: #f8d7da; color: #721c24"
    return ""

def color_staleness(val):
    if isinstance(val, (int, float)):
        if val > MAX_AGE_DAYS:
            return "background-color: #f8d7da; color: #721c24"
        elif val > MAX_AGE_DAYS // 2:
            return "background-color: #fff3cd; color: #856404"
    return ""

display_cols = ["Series", "FRED Code", "Last Update", "Frequency",
                "Obs Count", "NaN Count", "Outliers",
                "Staleness (days)", "Status"]

styled = (summary_df[display_cols]
          .style
          .applymap(color_status, subset=["Status"])
          .applymap(color_staleness, subset=["Staleness (days)"])
          .set_caption("Data Quality Dashboard")
          .set_table_styles([
              {"selector": "caption",
               "props": [("font-size", "16px"), ("font-weight", "bold")]},
          ]))
styled

### Data Coverage Timeline

Horizontal bars show the date range covered by each series.
Red/yellow highlighting indicates stale or problematic series.

In [ ]:
# Visualize data coverage as horizontal timeline bars
fig, ax = plt.subplots(figsize=(14, max(6, len(series_data) * 0.5)))

y_labels = []
y_positions = []

for i, (display_name, (df, fred_code, expected_freq)) in enumerate(
        series_data.items()):
    s = df.iloc[:, 0] if isinstance(df, pd.DataFrame) else df
    if len(s) == 0:
        continue

    start = s.index[0]
    end = s.index[-1]
    staleness = (today - end).days

    # Color based on status
    if staleness > MAX_AGE_DAYS:
        color = "#dc3545"   # red - stale
        alpha = 0.9
    elif staleness > MAX_AGE_DAYS // 2:
        color = "#ffc107"   # yellow - warning
        alpha = 0.8
    else:
        color = "#28a745"   # green - fresh
        alpha = 0.7

    ax.barh(i, (end - start).days, left=mdates.date2num(start),
            height=0.6, color=color, alpha=alpha, edgecolor="white")

    # Annotate with date range
    ax.text(mdates.date2num(end) + 30, i,
            f"  {end.strftime('%Y-%m')}",
            va="center", fontsize=8, color="#333")

    y_labels.append(f"{display_name} ({fred_code})")
    y_positions.append(i)

ax.set_yticks(y_positions)
ax.set_yticklabels(y_labels, fontsize=9)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.xaxis.set_major_locator(mdates.YearLocator(5))
ax.set_xlabel("Date")
ax.set_title("Data Coverage Timeline", fontsize=14, fontweight="bold")
ax.invert_yaxis()
ax.grid(axis="x", alpha=0.3)

# Add legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor="#28a745", alpha=0.7, label="Fresh (< {0} days)".format(MAX_AGE_DAYS // 2)),
    Patch(facecolor="#ffc107", alpha=0.8, label="Aging ({0}-{1} days)".format(MAX_AGE_DAYS // 2, MAX_AGE_DAYS)),
    Patch(facecolor="#dc3545", alpha=0.9, label="Stale (> {0} days)".format(MAX_AGE_DAYS)),
]
ax.legend(handles=legend_elements, loc="lower right", fontsize=9)

plt.tight_layout()
plt.show()

### Detailed Issues

Series with warnings or errors are listed below for further investigation.

In [ ]:
# Show details for problematic series only
issues_df = summary_df[summary_df["Status"] != "PASS"][
    ["Series", "FRED Code", "Status", "Details"]
]

if len(issues_df) == 0:
    print("\u2713 All series passed validation with no issues.")
else:
    print(f"{len(issues_df)} series have warnings or errors:\n")
    for _, row in issues_df.iterrows():
        marker = "\u2717" if row["Status"] == "FAIL" else "\u26a0"
        print(f"  {marker} {row['Series']} ({row['FRED Code']})")
        for detail in row["Details"].split("; "):
            print(f"      {detail}")
        print()

---
*Dashboard generated using `fecon235.data_validation` module.
For documentation, see the [fecon235 repository](https://github.com/rsvp/fecon235).*